# AnyMani pre-made connectivity quick notebook

本 notebook 记录 **pre-made 打通阶段** 的最短可运行指令。

- 默认目标：同时枚举 `single_palm_allegro` 与 `single_palm_leap` 的**全部合法 connectivity 变体**
- 若只想看单个 family：删掉 `hand_presets` 里另一个 hand preset 即可
- 若只想看一小部分 connectivity：把 `connectivity_presets=None` 改成显式映射即可

# NOTE
当前 pre-made connectivity 的科学语义已经对齐为：
**删除 joint = 删除该 joint 所代表的 child-link 几何节点，并对 surviving 链做自动重解算。**


In [5]:
import sys
from pathlib import Path

# 约定优先从仓库根目录启动 notebook；若不是，则回退到当前固定仓库路径。
repo_root = Path.cwd()
if not (repo_root / "source" / "anymani").exists():
    repo_root = Path("/home/hac/isaac/AnyMani")  # VS Code 若不是从仓库根打开，则显式回退

# 让 `import anymani...` 直接可用。
sys.path.insert(0, str(repo_root / "source" / "anymani"))

from anymani.assets.generator.hand_generator import HandGenerator, HandGeneratorCfg
from anymani.assets.presets.connectivity_presets import list_hand_connectivity_preset_names

# 正式 generated 根目录；all-legal enumerate 产物会按 recursive 布局落在这里。
output_dir = repo_root / "source" / "anymani" / "anymani" / "assets" / "generated"

# 默认同时跑 Allegro + LEAP；若只想看单个 family，删掉其中一项即可。
hand_presets = ["single_palm_allegro", "single_palm_leap"]

output_dir


PosixPath('/home/hac/isaac/AnyMani/source/anymani/anymani/assets/generator/source/anymani/anymani/assets/generated')

In [6]:
# 先把当前 registry 中所有合法 connectivity 名字完整列出来，便于人工巡检与挑选。
for family in ("allegro", "leap"):
    names = list_hand_connectivity_preset_names(family)
    print(f"=== {family} ({len(names)} variants) ===")
    for name in names:
        print(name)
    print()


=== allegro (54 variants) ===
allegro_full
allegro_t3_i2_m2_r2
allegro_t3_i2_m2_r3
allegro_t3_i2_m2_r4
allegro_t3_i2_m3_r2
allegro_t3_i2_m3_r3
allegro_t3_i2_m3_r4
allegro_t3_i2_m4_r2
allegro_t3_i2_m4_r3
allegro_t3_i2_m4_r4
allegro_t3_i3_m2_r2
allegro_t3_i3_m2_r3
allegro_t3_i3_m2_r4
allegro_t3_i3_m3_r2
allegro_t3_i3_m3_r3
allegro_t3_i3_m3_r4
allegro_t3_i3_m4_r2
allegro_t3_i3_m4_r3
allegro_t3_i3_m4_r4
allegro_t3_i4_m2_r2
allegro_t3_i4_m2_r3
allegro_t3_i4_m2_r4
allegro_t3_i4_m3_r2
allegro_t3_i4_m3_r3
allegro_t3_i4_m3_r4
allegro_t3_i4_m4_r2
allegro_t3_i4_m4_r3
allegro_t3_i4_m4_r4
allegro_t4_i2_m2_r2
allegro_t4_i2_m2_r3
allegro_t4_i2_m2_r4
allegro_t4_i2_m3_r2
allegro_t4_i2_m3_r3
allegro_t4_i2_m3_r4
allegro_t4_i2_m4_r2
allegro_t4_i2_m4_r3
allegro_t4_i2_m4_r4
allegro_t4_i3_m2_r2
allegro_t4_i3_m2_r3
allegro_t4_i3_m2_r4
allegro_t4_i3_m3_r2
allegro_t4_i3_m3_r3
allegro_t4_i3_m3_r4
allegro_t4_i3_m4_r2
allegro_t4_i3_m4_r3
allegro_t4_i3_m4_r4
allegro_t4_i4_m2_r2
allegro_t4_i4_m2_r3
allegro_t4_i4_m2_

In [7]:
# 这是当前 pre-made 打通阶段的主入口：
# - `hand_presets` 同时给 Allegro + LEAP
# - `connectivity_presets=None` 表示：
#   对每个 hand preset 自动展开其所属 family 的全部合法 connectivity 变体
# - `artifact_level="bundle"` 让 URDF / sidecar 一起落盘，便于你直接用 URDF 插件巡检
generator = HandGenerator(
    HandGeneratorCfg(
        mode="made",
        artifact_level="bundle",
        output_dir=output_dir,
        sampling_strategy="enumerate",
        hand_presets=hand_presets,
        connectivity_presets=None,
        output_layout="recursive",
    )
)

results = list(generator.generate_batch())
print(f"generated {len(results)} bundles under {output_dir}")
for result in results:
    print(result.metadata["base_hand_preset"], result.metadata["connectivity_preset"], result.urdf_path)


TypeError: HandGeneratorCfg.__init__() got an unexpected keyword argument 'hand_presets'

In [ ]:
# 如果你只想看某几个 connectivity，可以把 `connectivity_presets` 改成显式映射。
# 这里给一个最小示例：每个 family 只跑 full + 一个 reduced connectivity。
subset_generator = HandGenerator(
    HandGeneratorCfg(
        mode="made",
        artifact_level="bundle",
        output_dir=output_dir,
        sampling_strategy="enumerate",
        hand_presets=["single_palm_allegro", "single_palm_leap"],
        connectivity_presets={
            "single_palm_allegro": ["allegro_full", "allegro_t3_i2_m2_r2"],
            "single_palm_leap": ["leap_full", "leap_t3_i2_m2_r2"],
        },
        output_layout="recursive",
    )
)

subset_results = list(subset_generator.generate_batch())
print(f"generated {len(subset_results)} subset bundles")
for result in subset_results:
    print(result.metadata["base_hand_preset"], result.metadata["connectivity_preset"], result.urdf_path)
